In [4]:
from datasets import load_dataset
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from transformers import pipeline
import evaluate
import numpy as np

# 1. Load dataset
dataset = load_dataset("imdb")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
    
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
eval_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(train_dataset[0])

# Load the tokenizer associated with the pre-trained model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Define a tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply the tokenization to our entire datasets using map()
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# Load the model and specify we have 2 labels (Positive and Negative)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Get the prediction with the highest probability
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Define the training hyperparameters
training_args = TrainingArguments(
    output_dir="./bert-sentiment-results", # Where to save the model
    eval_strategy="epoch",                 # Evaluate at the end of each epoch
    learning_rate=2e-5,                    # Standard learning rate for fine-tuning
    per_device_train_batch_size=8,         # Adjust based on your GPU VRAM
    per_device_eval_batch_size=8,
    num_train_epochs=3,                    # 3 epochs is usually plenty for BERT
    weight_decay=0.01,                     # Helps prevent overfitting
    logging_dir='./logs',
)

# Assemble the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# Start the training process
trainer.train()


# Load your fine-tuned model into a pipeline for easy inference
# We'll map the numerical labels back to readable text
classifier = pipeline(
    "sentiment-analysis", 
    model=model, 
    tokenizer=tokenizer,
    device=0 # 0 sets it to use the GPU. Remove if on CPU.
)

# Test it out!
test_reviews = [
    "I absolutely loved this movie, the acting was phenomenal!",
    "What a waste of time. The plot was incredibly boring and made no sense."
]

results = classifier(test_reviews)

for review, result in zip(test_reviews, results):
    label = "Positive" if result['label'] == 'LABEL_1' else "Negative"
    print(f"Review: '{review}'")
    print(f"Prediction: {label} (Score: {result['score']:.4f})\n")

Using device: cuda
{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `loggin

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.276692,0.908000
2,0.296239,0.516505,0.870000
3,0.296239,0.429274,0.914000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Review: 'I absolutely loved this movie, the acting was phenomenal!'
Prediction: Positive (Score: 0.9982)

Review: 'What a waste of time. The plot was incredibly boring and made no sense.'
Prediction: Negative (Score: 0.9976)

